In [34]:
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
torch.manual_seed(33)
torch.cuda.manual_seed_all(33)

In [35]:
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False


def _find_project_root():
    # 1) Prefer current notebook working dir and its parents.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "configs" / "config.yaml").exists():
            return candidate

    # 2) Colab fallback: look for repo folders under common mount points.
    if IS_COLAB:
        for base in (Path("/content"), Path("/content/drive/MyDrive")):
            if not base.exists():
                continue

            if (base / "configs" / "config.yaml").exists():
                return base

            for child in base.iterdir():
                if child.is_dir() and (child / "configs" / "config.yaml").exists():
                    return child

    return Path("/content") if IS_COLAB else Path.cwd()


def _merge_dicts(base: dict, override: dict) -> dict:
    merged = dict(base)
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _merge_dicts(merged[key], value)
        else:
            merged[key] = value
    return merged


BASE_DIR = _find_project_root()
DATA_ROOT = BASE_DIR / "data"
CHECKPOINT_DIR = BASE_DIR / "results" / "checkpoints"

# Toggle this to choose notebook defaults or configs/config.yaml.
USE_CONFIG_YAML = False

default_config = {
    "training": {
        "epochs": 40,
        "batch_size": 256,
        "learning_rate": 0.01,
        "momentum": 0.9,
        "weight_decay": 5e-4,
    },
    "testing": {"batch_size": 256},
    "train_dataloader": {"num_workers": 6},
    "test_dataloader": {"num_workers": 6},
    "scheduler": {
        "type": "step",
        "step_size": 15,
        "gamma": 0.1,
        "milestones": [30, 45],
    },
    "model_to_save": {"name": "ResNet18_notebook.pth"},
    "model_to_test": {"name": "ResNet18_notebook.pth"},
}

_yaml_path = BASE_DIR / "configs" / "config.yaml"
config = default_config
if USE_CONFIG_YAML and _yaml_path.exists():
    import yaml
    with open(_yaml_path, "r", encoding="utf-8") as _f:
        yaml_config = yaml.safe_load(_f) or {}
    config = _merge_dicts(default_config, yaml_config)

save_model_name = (
    config.get("model_to_save", {}).get("name")
    or config.get("model", {}).get("name")
    or "ResNet18_notebook.pth"
)
test_model_name = config.get("model_to_test", {}).get("name") or save_model_name

SAVE_MODEL_PATH = CHECKPOINT_DIR / save_model_name
TEST_MODEL_PATH = CHECKPOINT_DIR / test_model_name

In [36]:
import torch.nn as nn


class BasicBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        downsample: nn.Module = None,
    ):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1
        )
        self.batchnorm1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample

    def forward(self, x):
        inp = x
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.batchnorm2(x)
        if self.downsample is not None:
            inp = self.downsample(inp)
        x = x + inp  # skip connection implementation
        x = self.relu(x)
        return x


class ResNet18(nn.Module):
    def __init__(self, num_classes: int):
        super(ResNet18, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.batchnorm1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # Define the layers of ResNet-18
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(
        self, in_channels: int, out_channels: int, num_blocks: int, stride: int = 1
    ):
        layers = []
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels),
            )

        layers.append(BasicBlock(in_channels, out_channels, stride, downsample))
        in_channels = out_channels

        for _ in range(1, num_blocks):
            layers.append(BasicBlock(in_channels, out_channels))
            in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x


In [37]:
from torchvision.transforms import v2
from torchvision.datasets import CIFAR10
import torch
def get_datasets():

    train_transform = v2.Compose(
        [
            v2.RandomCrop(32, padding=4),
            v2.RandomHorizontalFlip(),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ]
    )
    test_transform = v2.Compose(
        [
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ]
    )
    train_dataset = CIFAR10(
        root=str(DATA_ROOT), train=True, download=True, transform=train_transform
    )

    test_dataset = CIFAR10(
        root=str(DATA_ROOT), train=False, download=True, transform=test_transform
    )

    return train_dataset, test_dataset


In [38]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def use_pin_memory(device, workers):
    return device.type == "cuda" and not (os.name == "nt" and workers > 0)


def make_data_loader(dataset, batch_size, workers, shuffle, device):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=use_pin_memory(device, workers),
        persistent_workers=workers > 0,
    )

In [39]:
training_config = config["training"]
train_dataloader_config = config["train_dataloader"]
scheduler_config = config.get("scheduler", {})


def train():
    torch.set_float32_matmul_precision("high")

    device = get_device()
    print(f"Using device: {device}")
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    train_dataset, _ = get_datasets()
    train_loader = make_data_loader(
        train_dataset,
        batch_size=training_config["batch_size"],
        workers=train_dataloader_config["num_workers"],
        shuffle=True,
        device=device,
    )

    model = ResNet18(num_classes=10).to(device)
    if hasattr(torch, "compile") and device.type == "cuda":
        model = torch.compile(model)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=training_config["learning_rate"],
        momentum=training_config["momentum"],
        weight_decay=training_config["weight_decay"],
        
    )

    scheduler_type = scheduler_config.get("type", "step")
    if scheduler_type == "step":
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=scheduler_config.get("step_size", 15),
            gamma=scheduler_config.get("gamma", 0.1),
        )
    elif scheduler_type == "multistep":
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=scheduler_config.get("milestones", [30, 45]),
            gamma=scheduler_config.get("gamma", 0.1),
        )
    else:
        scheduler = None

    prev_lr = optimizer.param_groups[0]["lr"]

    for epoch in tqdm(range(training_config["epochs"]), desc="Training epochs"):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, targets in train_loader:
            non_blocking = use_pin_memory(device, train_dataloader_config["num_workers"])
            inputs = inputs.to(device, non_blocking=non_blocking)
            targets = targets.to(device, non_blocking=non_blocking)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            predictions = torch.argmax(outputs, dim=1)
            train_correct += (predictions == targets).sum().item()
            train_total += targets.size(0)

        epoch_loss = train_loss / train_total
        epoch_accuracy = train_correct / train_total
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch [{epoch + 1}/{training_config['epochs']}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}, LR: {current_lr:.5f}"
        )

        if scheduler is not None:
            scheduler.step()
            new_lr = optimizer.param_groups[0]["lr"]
            if new_lr != prev_lr:
                print(f"LR changed -> {new_lr:.5f}")
                prev_lr = new_lr

        model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model
        torch.save(model_to_save.state_dict(), SAVE_MODEL_PATH)


In [40]:
testing_config = config["testing"]
test_dataloader_config = config["test_dataloader"]


def test(model_name=None):
    device = get_device()
    _, test_dataset = get_datasets()

    model_path = TEST_MODEL_PATH if not model_name else (CHECKPOINT_DIR / model_name)
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found at {model_path}. Train first or provide a valid model filename."
        )

    test_loader = make_data_loader(
        test_dataset,
        batch_size=testing_config["batch_size"],
        workers=test_dataloader_config["num_workers"],
        shuffle=False,
        device=device,
    )

    model = ResNet18(num_classes=10).to(device)
    model.load_state_dict(
        torch.load(model_path, map_location=device, weights_only=True)
    )
    model.eval()

    correct = 0
    total = 0
    non_blocking = use_pin_memory(device, test_dataloader_config["num_workers"])

    with torch.inference_mode():
        for inputs, targets in tqdm(test_loader, desc="Testing"):
            inputs = inputs.to(device, non_blocking=non_blocking)
            targets = targets.to(device, non_blocking=non_blocking)
            outputs = model(inputs)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

    test_accuracy = correct / total
    print(f"Model: {model_path.name}")
    print(f"Test Accuracy: {test_accuracy:.4f}")

In [41]:
# train()
# test()

In [43]:
import random
import gradio as gr
import torch
from torchvision.transforms import v2


_infer_model = None
_infer_device = None
_infer_model_path = None
_val_dataset = None


def _get_infer_resources():
    global _infer_model, _infer_device, _infer_model_path, _val_dataset
    model_path = TEST_MODEL_PATH

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found at {model_path}. Train first or update model_to_test in config."
        )

    if _infer_model is None or _infer_model_path != model_path:
        _infer_device = get_device()
        _infer_model = ResNet18(num_classes=10).to(_infer_device)
        _infer_model.load_state_dict(
            torch.load(model_path, map_location=_infer_device, weights_only=True)
        )
        _infer_model.eval()
        _infer_model_path = model_path
        if _val_dataset is None:
            _, _val_dataset = get_datasets()

    return _infer_model, _infer_device, _val_dataset, _infer_model_path


_mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
_std = torch.tensor((0.2023, 0.1994, 0.2010)).view(3, 1, 1)


def _to_display_image(image_tensor: torch.Tensor):
    image = image_tensor.detach().cpu()
    image = image * _std + _mean
    image = image.clamp(0.0, 1.0)
    return v2.ToPILImage()(image)


def predict_random_five():
    model, device, val_dataset, model_path = _get_infer_resources()
    class_names = val_dataset.classes
    sample_count = min(5, len(val_dataset))
    indices = random.sample(range(len(val_dataset)), sample_count)
    gallery_items = []

    for idx in indices:
        image_tensor, true_label = val_dataset[idx]
        tensor = image_tensor.unsqueeze(0).to(device)

        with torch.inference_mode():
            probs = torch.softmax(model(tensor), dim=1)
            top_prob, top_idx = probs[0].max(0)

        pred_class = class_names[top_idx.item()]
        true_class = class_names[true_label]
        caption = (
            f"Predicted: {pred_class}  ({top_prob.item() * 100:.1f}%)\n"
            f"True label: {true_class}"
        )
        gallery_items.append((_to_display_image(image_tensor), caption))

    return gallery_items


with gr.Blocks(title="ResNet-18 · CIFAR-10 Predictor") as demo:
    gr.Markdown(
        "## ResNet-18 · Random 5-Image Predictor\n"
    )
    sample_btn = gr.Button("Sample & Predict", variant="primary")
    gallery = gr.Gallery(
        label="Predictions",
        columns=5,
        rows=1,
        height="auto",
        object_fit="contain",
        show_label=True,
    )
    sample_btn.click(fn=predict_random_five, outputs=gallery)

demo.launch(share=IS_COLAB)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
